# Build environment
- The code in this notebook has been integrated into `tools.create_2D_ocean_slice`

In [ ]:
import os

import gsw
import matplotlib.pyplot as plt
import numpy as np
import pygenray as pr
import xarray as xr
import xrsignal as xrs
from dotenv import find_dotenv, load_dotenv
from scipy import io, signal

from nescafe23_fluctuations import tools

_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
coords = {
    "SION": [39 + 37.194 / 60, -63 - 39.427 / 60],
    "SIOE": [38 + 45.906 / 60, -62 - 15.626 / 60],
}

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/SION_CTD.mat"
SION_mat = io.loadmat(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/SIOE_CTD.mat"
SIOE_mat = io.loadmat(fn)

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/FCTD_SoundSpeed_Colosi_v2.mat"
fctd_mat = io.loadmat(fn)

fctds = {}
fctds_pilot = {}

deep_value = {"tt": 2.3, "ss": 34.885}  # , 'cc':1640}

for k, var in enumerate(["tt", "ss"]):
    fctds[var] = xr.DataArray(
        fctd_mat[var],
        dims=["range", "depth"],
        coords={
            "range": fctd_mat["r"].flatten() * 1000,
            "depth": fctd_mat["z"].flatten(),
            "latitude": ("range", fctd_mat["lat"].flatten()),
            "longitude": ("range", fctd_mat["lon"].flatten()),
        },
    ).sortby("range")

    # Add 10000 meter
    fctds_pilot[var] = xr.concat(
        (
            fctds[var],
            xr.ones_like(fctds[var][:, -2:-1]).assign_coords({"depth": [10000]})
            * deep_value[var],
        ),
        dim="depth",
    )

fctds_pilot = xr.Dataset(fctds_pilot)

In [ ]:
# interpolate to Δr=200m, Δz = 3km
fctd_grid = fctds_pilot.interp({
    "range": np.arange(0, fctds_pilot.range[-1], 500),
    "depth": np.arange(0, 6000),
})

r_smooth = 3000
z_smooth = 200

bz, az = signal.butter(4, 1 / z_smooth, fs=1)
br, ar = signal.butter(4, 1 / r_smooth, fs=1 / 500)

front = {}
for k, var in enumerate(["tt", "ss"]):
    fctd_smooth = xrs.filtfilt(
        xrs.filtfilt(fctd_grid[var], dim="range", b=br, a=ar), dim="depth", b=bz, a=az
    )

    front[var] = fctd_smooth.sel({"range": slice(60000, 170000)})
    front[var] = front[var].assign_coords({"range": front[var].range.values - 115000})
    front[var] = (
        front[var]
        .assign_coords({"range": np.flip(front[var].range.values)})
        .sortby("range")
    )

    front[var] = front[var].drop_vars(["latitude", "longitude"])

front = xr.Dataset(front)

In [ ]:
full_field_range_coord = np.arange(-155000, 155000, 500)

mask = xr.DataArray(
    np.ones((6000, len(full_field_range_coord))),
    dims=["depth", "range"],
    coords={"depth": front.depth, "range": full_field_range_coord},
)

In [ ]:
front_rep = xr.concat(
    (
        front.isel({"range": 0}).assign_coords({"range": -155000}),
        front,
        front.isel({"range": -1}).assign_coords({"range": 155000}),
    ),
    dim="range",
)

In [ ]:
front_full = front_rep.interp_like(
    mask, kwargs={"bounds_error": False, "fill_value": "extrapolate"}
)

In [ ]:
SION = {
    "tt": xr.DataArray(
        SION_mat["t5"].flatten(),
        dims=["depth"],
        coords={"depth": SION_mat["z5"].flatten()},
    ),
    "ss": xr.DataArray(
        SION_mat["s5"].flatten(),
        dims=["depth"],
        coords={"depth": SION_mat["z5"].flatten()},
    ),
}

SIOE = {
    "tt": xr.DataArray(
        SIOE_mat["t8"].flatten(),
        dims=["depth"],
        coords={"depth": SIOE_mat["z8"].flatten()},
    ),
    "ss": xr.DataArray(
        SIOE_mat["s8"].flatten(),
        dims=["depth"],
        coords={"depth": SIOE_mat["z8"].flatten()},
    ),
}

for var in SION.keys():
    # Add 10000 meter
    SION[var] = xr.concat(
        (
            SION[var],
            xr.ones_like(SION[var][-2:-1]).assign_coords({"depth": [10000]})
            * deep_value[var],
        ),
        dim="depth",
    )

    # Add 10000 meter
    SIOE[var] = xr.concat(
        (
            SIOE[var],
            xr.ones_like(SIOE[var][-2:-1]).assign_coords({"depth": [10000]})
            * deep_value[var],
        ),
        dim="depth",
    )

    SION[var] = xrs.filtfilt(SION[var], dim="depth", b=bz, a=az)
    SIOE[var] = xrs.filtfilt(SIOE[var], dim="depth", b=bz, a=az)


SION = xr.Dataset(SION)
SIOE = xr.Dataset(SIOE)

In [ ]:
SION_full = SION.interp_like(
    mask, kwargs={"bounds_error": False, "fill_value": "extrapolate"}
).broadcast_like(mask)

SIOE_full = SIOE.interp_like(
    mask, kwargs={"bounds_error": False, "fill_value": "extrapolate"}
).broadcast_like(mask)

### remove local maximum from SION

In [ ]:
SION_full_nlm = SION_full.copy()
SION_full_nlm = SION_full_nlm.where(
    (SION_full_nlm.depth > 350) | (SION_full_nlm.depth < 25), np.nan
)
SION_full_nlm = SION_full_nlm.interpolate_na(dim="depth")

In [ ]:
fig = plt.figure(figsize=(10, 4))
fig.suptitle("Removing SION local maximum")

plt.subplot(1, 3, 1)
plt.plot(SION_full["tt"][:, 0], SION_full.depth)
plt.plot(SION_full_nlm["tt"][:, 0], SION_full_nlm.depth)
plt.title("In-Situ Temperature")
plt.xlabel("[$^\\circ$C]")
plt.ylim([3000, 0])
plt.ylabel("depth [m]")

plt.subplot(1, 3, 2)
plt.plot(SION_full["ss"][:, 0], SION_full.depth)
plt.plot(SION_full_nlm["ss"][:, 0], SION_full_nlm.depth)
plt.title("Practical Salinity")
plt.xlabel("[psu]")
plt.ylim([3000, 0])
plt.ylabel("depth [m]")


# compute C
p = gsw.p_from_z(-SION_full_nlm.depth, lat=coords["SION"][0])
c_SION_nm = tools.aog_vsdg(p, SION_full_nlm["tt"], SION_full_nlm["ss"])
c_SION = tools.aog_vsdg(p, SION_full["tt"], SION_full["ss"])

plt.subplot(1, 3, 3)
plt.plot(c_SION[:, 0], c_SION.depth, label="Smoothed CTD")
plt.plot(c_SION_nm[:, 0], c_SION_nm.depth, label="Smoothed CTD with\nlocal max removed")
plt.title("Sound Speed")
plt.xlabel("[m/s]")
plt.ylim([3000, 0])
plt.ylabel("depth [m]")
plt.legend(loc="lower right")

plt.tight_layout()

## Mix CTDs and FCTDS

In [ ]:
mask1 = mask.where((mask.depth < 800) & (mask.range > -50000) & (mask.range < 50000), 0)
mask2 = (-mask1 + 1).where(mask.range > -30000, 0)
mask3 = (-mask1 + 1).where(mask.range <= -30000, 0)

In [ ]:
fig = plt.figure(figsize=(10, 3))

plt.subplot(1, 3, 1)
mask1.plot()
plt.ylim([5000, 0])
plt.title("Mask FCTD")

plt.subplot(1, 3, 2)
mask2.plot()
plt.ylim([5000, 0])
plt.title("Mask SION")

plt.subplot(1, 3, 3)
mask3.plot()
plt.ylim([5000, 0])
plt.title("Mask SIOE")

plt.tight_layout()

In [ ]:
z = np.linspace(-np.pi, np.pi, 600)
x = np.linspace(-np.pi, np.pi, 120)

kernel_x = (np.cos(x) + 1) / 2
kernel_z = (np.cos(z) + 1) / 2

kernel = kernel_z[:, None] * kernel_x[None, :]
kernel = kernel / np.sum(kernel)

In [ ]:
def fftconvolve_mirror(input_array, kernel, mode="same"):
    """
    FFT convolution with mirror boundary conditions.

    Parameters:
    -----------
    input_array : array_like
        Input array to convolve
    kernel : array_like
        Convolution kernel
    mode : str
        'same', 'valid', or 'full'
    """
    input_shape = np.array(input_array.shape)
    kernel_shape = np.array(kernel.shape)

    # Calculate padding needed (kernel_size - 1 on each side)
    pad_width = [(k - 1, k - 1) for k in kernel_shape]

    # Pad with mirror/reflection
    padded = np.pad(input_array, pad_width, mode="reflect")

    # Perform FFT convolution with mode='valid'
    result = signal.fftconvolve(padded, kernel, mode="valid")

    if mode == "same":
        # Center the result by cropping from the correct starting position
        # The valid convolution on padded input produces an output that needs to be
        # cropped by (k-1)//2 from the start to align with scipy's 'same' mode
        start = [(k - 1) // 2 for k in kernel_shape]
        slices = tuple(slice(s, s + inp_s) for s, inp_s in zip(start, input_shape))
        return result[slices]
    elif mode == "full":
        return result
    elif mode == "valid":
        # Calculate valid region size
        valid_shape = input_shape - kernel_shape + 1
        # Center the valid region using the same offset as 'same' mode
        start = [(k - 1) // 2 for k in kernel_shape]
        slices = tuple(slice(s, s + v) for s, v in zip(start, valid_shape))
        return result[slices]

    return result

In [ ]:
mask1_smooth = xr.DataArray(
    fftconvolve_mirror(mask1.values, kernel, mode="same"),
    dims=["depth", "range"],
    coords={"depth": mask.depth, "range": mask.range},
)

mask2_smooth = xr.DataArray(
    fftconvolve_mirror(mask2.values, kernel, mode="same"),
    dims=["depth", "range"],
    coords={"depth": mask.depth, "range": mask.range},
)

mask3_smooth = xr.DataArray(
    fftconvolve_mirror(mask3.values, kernel, mode="same"),
    dims=["depth", "range"],
    coords={"depth": mask.depth, "range": mask.range},
)

In [ ]:
fig = plt.figure(figsize=(10, 3))

plt.subplot(1, 3, 1)
mask1_smooth.plot()
plt.ylim([5000, 0])
plt.title("Mask FCTD")

plt.subplot(1, 3, 2)
mask2_smooth.plot()
plt.ylim([5000, 0])
plt.title("Mask SIOE")

plt.subplot(1, 3, 3)
mask3_smooth.plot()
plt.ylim([5000, 0])
plt.title("Mask SION")

plt.tight_layout()

In [ ]:
full_field = (
    front_full * mask1_smooth + SION_full * mask2_smooth + SIOE_full * mask3_smooth
)

full_field["tt"].attrs["units"] = "deg C"
full_field["tt"].name = "in-situ temperature"

full_field["ss"].attrs["units"] = "psu"
full_field["ss"].name = "pracitical salinity"

full_field_nlm = (
    front_full * mask1_smooth + SION_full_nlm * mask2_smooth + SIOE_full * mask3_smooth
)

full_field_nlm["tt"].attrs["units"] = "deg C"
full_field_nlm["tt"].name = "in-situ temperature"

full_field_nlm["ss"].attrs["units"] = "psu"
full_field_nlm["ss"].name = "pracitical salinity"

## Calculate all oceanographic variables

In [ ]:
p = gsw.p_from_z(-full_field.depth, coords["SION"][0])

## Salinity
SA = gsw.SA_from_SP(full_field["ss"], p, coords["SION"][1], coords["SION"][0])
SA.attrs["units"] = "g/kg"
SA.name = "Absolute Salinity"

SA_nlm = gsw.SA_from_SP(full_field_nlm["ss"], p, coords["SION"][1], coords["SION"][0])
SA_nlm.attrs["units"] = "g/kg"
SA_nlm.name = "Absolute Salinity"

## Conservative Temperature
CT = gsw.CT_from_t(SA, full_field["tt"], p)
CT.attrs["units"] = "deg C"
CT.name = "Conservative Temperature"

CT_nlm = gsw.CT_from_t(SA_nlm, full_field_nlm["tt"], p)
CT_nlm.attrs["units"] = "deg C"
CT_nlm.name = "Conservative Temperature"

## Sound Spped
c = tools.aog_vsdg(p, full_field["tt"], full_field["ss"])
c.attrs["units"] = "m/s"
c.name = "Sound Speed"

c_nlm = tools.aog_vsdg(p, full_field_nlm["tt"], full_field_nlm["ss"])
c_nlm.attrs["units"] = "m/s"
c_nlm.name = "Sound Speed"

## Buoyancy Frequency
N2, pmid = gsw.Nsquared(SA, CT, p.values[:, None], lat=coords["SION"][0], axis=0)
z_mid = gsw.z_from_p(pmid, coords["SION"][0])[:, 0]
N2x = xr.DataArray(
    N2, dims=["depth", "range"], coords={"depth": -z_mid, "range": full_field.range}
)
N = (
    np.sqrt(
        N2x.interp(
            {"depth": full_field.depth},
            kwargs={"bounds_error": False, "fill_value": "extrapolate"},
        )
    )
    / (2 * np.pi)
    * 3600
)
N.attrs["units"] = "cph"
N.name = "Buoyancy Frequency"

N2_nlm, pmid = gsw.Nsquared(
    SA_nlm, CT_nlm, p.values[:, None], lat=coords["SION"][0], axis=0
)
z_mid = gsw.z_from_p(pmid, coords["SION"][0])[:, 0]
N2x_nlm = xr.DataArray(
    N2_nlm, dims=["depth", "range"], coords={"depth": -z_mid, "range": full_field.range}
)
N_nlm = (
    np.sqrt(
        N2x_nlm.interp(
            {"depth": full_field.depth},
            kwargs={"bounds_error": False, "fill_value": "extrapolate"},
        )
    )
    / (2 * np.pi)
    * 3600
)
N_nlm.attrs["units"] = "cph"
N_nlm.name = "Buoyancy Frequency"

## Potential Temperature
PT = gsw.pt_from_t(SA, full_field["tt"], p, 0)
PT.attrs["units"] = "deg C"
PT.name = "potential temperature"

PT_nlm = gsw.pt_from_t(SA_nlm, full_field_nlm["tt"], p, 0)
PT_nlm.attrs["units"] = "deg C"
PT_nlm.name = "potential temperature"

## Adiabatic Sound Speed Gradient
C_adiabatic = tools.aog_vsdg(0, PT, full_field["ss"])
C_adiabatic.attrs["units"] = "deg C"
C_adiabatic.name = "adiabatic sound speed"

C_adiabatic_nlm = tools.aog_vsdg(0, PT_nlm, full_field_nlm["ss"])
C_adiabatic_nlm.attrs["units"] = "deg C"
C_adiabatic_nlm.name = "adiabatic sound speed"

dcdz_p = -C_adiabatic.differentiate("depth")
dcdz_p.attrs["units"] = "[m/s^2]"
dcdz_p.name = "adiabatic vertical sound speed gradient"

dcdz_p_nlm = -C_adiabatic_nlm.differentiate("depth")
dcdz_p_nlm.attrs["units"] = "[m/s^2]"
dcdz_p_nlm.name = "adiabatic vertical sound speed gradient"

In [ ]:
full_field_full = xr.Dataset({
    "t": full_field["tt"],
    "SP": full_field["ss"],
    "c": c,
    "N": N,
    "SA": SA,
    "dcdz_p": dcdz_p,
})
full_field_full_nlm = xr.Dataset({
    "t": full_field_nlm["tt"],
    "SP": full_field_nlm["ss"],
    "c": c_nlm,
    "N": N_nlm,
    "SA": SA_nlm,
    "dcdz_p": dcdz_p_nlm,
})

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dataproducts/full_field.nc"
full_field_full.to_netcdf(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dataproducts/full_field_nlm.nc"
full_field_full_nlm.to_netcdf(fn)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
axes = axes.flatten()

plt.sca(axes[0])
full_field_full["t"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("In-Situ Temperature")

plt.sca(axes[1])
full_field_full["SA"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("Absolute Salinity")

plt.sca(axes[2])
full_field_full["c"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("Sound Speed")

plt.sca(axes[3])
full_field_full["N"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("Buoyancy Frequency")

plt.sca(axes[4])
full_field_full_nlm["t"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("In-Situ Temperature")

plt.sca(axes[5])
full_field_full_nlm["SA"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("Absolute Salinity")

plt.sca(axes[6])
full_field_full_nlm["c"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("Sound Speed")

plt.sca(axes[7])
full_field_full_nlm["N"].assign_coords({"range": full_field_full.range / 1000}).plot()
plt.ylim([2000, 0])
plt.ylabel("depth [m]")
plt.xlabel("range [km]")
plt.title("Buoyancy Frequency")


plt.tight_layout()

## Look at ray runs through environments

In [ ]:
# open environments

# shift front location
front_location = 35e3

envi_field = full_field_full.assign_coords({
    "range": (full_field_full.range.values + front_location)
})
c = envi_field["c"]

envi_field_nlm = full_field_full_nlm.assign_coords({
    "range": (full_field_full_nlm.range.values + front_location)
})
c_nlm = envi_field_nlm["c"]

bathy = xr.DataArray(
    np.linspace(4908, 4908, 100),
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)

envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)
envi_nlm = pr.OceanEnvironment2D(sound_speed=c_nlm, bathymetry=bathy)

In [ ]:
# Found with source localization
source_depth = 2361.50872806
source_range = 0
launch_angles = np.linspace(-30, 30, 8000)
receiver_range = 150404.34909747
num_range_save = 1000
receiver_depths = [999.37979522]

In [ ]:
rays = pr.shoot_rays(
    source_depth, source_range, launch_angles, receiver_range, num_range_save, envi
)
eigs = pr.find_eigenrays(
    rays,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi,
    ztol=30,
    max_iter=75,
)

rays_nlm = pr.shoot_rays(
    source_depth, source_range, launch_angles, receiver_range, num_range_save, envi_nlm
)
eigs_nlm = pr.find_eigenrays(
    rays_nlm,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi_nlm,
    ztol=30,
    max_iter=75,
)

In [ ]:
rays.plot_time_front(c="C0", add_colorbar=False, label="Original")
rays_nlm.plot_time_front(c="C1", add_colorbar=False, label="Local Max\nRemoved")

plt.legend(markerscale=5)
plt.xlim([100, 102])
plt.xlabel("time [s]")
plt.ylabel("depth [m]")

## Look at Ray seperation

In [ ]:
ray_idx = np.where(eigs.ray_id[0] == "6.0")[0][0]
ray_idx_nlm = np.where(eigs_nlm.ray_id[0] == "6.0")[0][0]

In [ ]:
eigs.plot_ducted(c="C0")
eigs_nlm.plot_ducted(c="C1")

plt.ylim([5000, 0])

In [ ]:
launch_angle = eigs.launch_angles[0][ray_idx]
launch_angle_nlm = eigs_nlm.launch_angles[0][ray_idx_nlm]

launch_angle, launch_angle_nlm

In [ ]:
rays_eps = pr.shoot_rays(
    source_depth,
    source_range,
    [launch_angle, launch_angle + np.degrees(1e-5)],
    receiver_range,
    num_range_save,
    envi,
)
rays_eps_nlm = pr.shoot_rays(
    source_depth,
    source_range,
    [launch_angle_nlm, launch_angle_nlm + np.degrees(1e-5)],
    receiver_range,
    num_range_save,
    envi_nlm,
)

In [ ]:
ray_idx = np.where(eigs.ray_id[0] == "-6.0")[0][0]
launch_angle = eigs.launch_angles[0][ray_idx]
ray_m6 = pr.shoot_ray(
    source_depth, source_range, launch_angle, receiver_range, num_range_save, envi
)
ray_m6_eps = pr.shoot_ray(
    source_depth,
    source_range,
    launch_angle + np.degrees(1e-5),
    receiver_range,
    num_range_save,
    envi,
)

zeta1 = ray_m6.z - ray_m6_eps.z

ray_idx_nlm = np.where(eigs_nlm.ray_id[0] == "-6.0")[0][0]
launch_angle_nlm = eigs_nlm.launch_angles[0][ray_idx_nlm]
ray_m6_nlm = pr.shoot_ray(
    source_depth,
    source_range,
    launch_angle_nlm,
    receiver_range,
    num_range_save,
    envi_nlm,
)
ray_m6_nlm_eps = pr.shoot_ray(
    source_depth,
    source_range,
    launch_angle_nlm + np.degrees(1e-5),
    receiver_range,
    num_range_save,
    envi_nlm,
)

zeta1 = ray_m6.z - ray_m6_eps.z
zeta1 = zeta1 / zeta1[-1]

zeta1_nlm = ray_m6_nlm.z - ray_m6_nlm_eps.z
zeta1_nlm = zeta1_nlm / zeta1_nlm[-1]

In [ ]:
plt.plot(ray_m6.r / 1000, zeta1, label="Original\nEnvironment")
plt.plot(ray_m6_nlm.r / 1000, zeta1_nlm, label="Local Max\nRemoved")

plt.title("Ray Seperation")
plt.xlabel("range [km]")
plt.legend()

In [ ]:
ray_m6.plot(label="Original\nEnvironment")
ray_m6_nlm.plot(label="Local Max\nRemoved")


plt.ylim([-5000, 0])
# plt.ylim([-500,0])
plt.xlabel("range [km]")
plt.legend()